In [24]:
# chat models, create_agent, message streaming
# basic prompting, few shot examples, structured promting, structured response(pydantic base model), system prompt
# tools
# memory

from dotenv import load_dotenv

load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel, Field
from langchain.tools import tool
import time
import sys
from connect_anki import request_anki

In [25]:
class KanjiExample(BaseModel):
    word: str = Field(description="The example word containing the kanji (e.g., 火山)")
    kana: str = Field(description="The reading in hiragana or katakana (e.g., かざん)")
    romaji: str = Field(description="The romaji reading (e.g., kazan)")
    meaning: str = Field(description="The English meaning of the word (e.g., volcano)")

    def __str__(self):
        return f"{self.word} [{self.kana}] ({self.romaji}) - {self.meaning}"

class KanjiFormat(BaseModel):
    onyomi: str = Field(description="The On'yomi reading(s) in katakana and romaji")
    kunyomi: str = Field(description="The Kunyomi reading(s) in hiragana and romaji")
    kanji_meaning: str = Field(description="The English meaning of the given Kanji")
    onyomi_examples: list[KanjiExample] = Field(description="List of example words using the On'yomi reading")
    kunyomi_examples: list[KanjiExample] = Field(description="List of example words using the Kunyomi reading")
    
    def to_polished_string(self) -> str:
        onyomi_ex_str = ", ".join(str(ex) for ex in self.onyomi_examples)
        kunyomi_ex_str = ", ".join(str(ex) for ex in self.kunyomi_examples)
        
        return (
            f"On'yomi: {self.onyomi}\n"
            f"Kunyomi: {self.kunyomi}\n"
            f"Kanji Meaning: {self.kanji_meaning}\n\n"
            f"Readings and Examples:\n"
            f"On'yomi: {onyomi_ex_str}\n"
            f"Kunyomi: {kunyomi_ex_str}"
        )

In [26]:
system_prompt = """
    You are a Japanese expert with 50 years of experience in teaching kanji. 
    Your role is to take a kanji as input and return its On'yomi and Kunyomi readings. 
    You must provide example words for both readings, including their kana, romaji, and English meanings.
    Then, use the `check_kanji_exists` tool to see if the user already has a card for it.
    If they do not, use the `create_kanji_flashcard` tool to add it to their deck.
"""

In [27]:
@tool("check_kanji_exists", description="Search Anki collection(deck) to see if a note for this specific kanji already exists.")
def check_kanji_exists(kanji: str) -> bool:
    params = {
        "query": f'deck:"Test_Deck1" "{kanji}"'
    }
    note_ids = request_anki(action="findNotes", params=params)
    return len(note_ids) > 0

In [28]:
@tool("create_kanji_flashcard", description="Use this tool to create kanji flashcards and add them to anki deck")
def create_kanji_flashcard(
    kanji: str, 
    onyomi: str, 
    kunyomi: str, 
    kanji_meaning: str, 
    onyomi_examples: list[KanjiExample], # Updated
    kunyomi_examples: list[KanjiExample] # Updated
) -> str:
    
    # 2. Because LangChain now knows they are KanjiExample objects, 
    # we can safely use dot notation (ex.word) instead of dictionary syntax (ex['word'])
    formatted_onyomi = [f"{ex.word} [{ex.kana}] ({ex.romaji}) - {ex.meaning}" for ex in onyomi_examples]
    formatted_kunyomi = [f"{ex.word} [{ex.kana}] ({ex.romaji}) - {ex.meaning}" for ex in kunyomi_examples]
    
    onyomi_ex_str = ", ".join(formatted_onyomi)
    kunyomi_ex_str = ", ".join(formatted_kunyomi)
    
    polished_text = (
        f"On'yomi: {onyomi}\n"
        f"Kunyomi: {kunyomi}\n"
        f"Kanji Meaning: {kanji_meaning}\n\n"
        f"Readings and Examples:\n"
        f"On'yomi: {onyomi_ex_str}\n"
        f"Kunyomi: {kunyomi_ex_str}"
    )
    
    # Convert standard newlines into HTML breaks for Anki
    html_back = polished_text.replace('\n', '<br>')
        
    params = {
        "note": {
            "deckName": "Test_Deck1",
            "modelName": "Basic",
            "fields": {
                "Front": kanji,
                "Back": html_back
            },
        }
    }
    
    response = request_anki(action="addNote", params=params)
    if response:
        return "success"
    return "unsuccess. Try again..."

In [29]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)

agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    response_format=KanjiFormat,
    tools=[check_kanji_exists, create_kanji_flashcard],
)

In [31]:

response = agent.invoke(
    {"messages": [HumanMessage(content="学")]}
)

parsed_response = response["structured_response"]
polished_text = parsed_response.to_polished_string()

for char in polished_text:
    sys.stdout.write(char)
    sys.stdout.flush()
    time.sleep(0.015)

On'yomi: ガク (GAKU)
Kunyomi: まな-ぶ (mana-bu)
Kanji Meaning: Study, Learning, Science

Readings and Examples:
On'yomi: 学校 [がっこう] (gakkou) - school, 学生 [がくせい] (gakusei) - student
Kunyomi: 学ぶ [まなぶ] (manabu) - to learn, to study